# 🌸 Nayari Dataset Builder  v3.8
**Run locally.** Scans `dataset/` (including subdirectories) for `.md`, `.txt`, `.pdf`, and `.json` files, applies a robust multi-format parser, deduplicates, quality-filters, and exports `nayari_dataset.json`.  Then auto-uploads to Kaggle.

| Feature | Details |
|---|---|
| Speaker formats | `Name:`, `**Name**:`, `> Name:`, `[Name]:` |
| Scene splitting | Blank-line gaps, `---END---`, `<end>`, `===` dividers |
| Auto-aliases | Unknown speakers scanned from filenames & content |
| Quality filter | Min 2 turns, min 10 chars/message |
| Deduplication | MD5 fingerprint on normalised content |
| OOC stripping | `[OOC: …]` and `(OOC …)` blocks removed |
| Source tagging | Every conversation records its origin file |
| Token estimate | Rough GPT-style token count in the stats cell |


## 📦 Step 0 — Install & Import

In [1]:
# 1. INSTALL DEPENDENCIES
# Note: pathlib is a built-in Python module and doesn't need pip installation.
# Added 'kaggle' to support the auto-upload features in the final steps.
%pip install -U openai pdfplumber kaggle -q

import sys
import json
import time
import re
import pdfplumber
import warnings
import shutil
import subprocess
import hashlib
from pathlib import Path
from openai import OpenAI
from collections import defaultdict
from datetime import datetime, timezone

# Suppress PDF-specific extraction warnings for a cleaner console
warnings.filterwarnings("ignore", category=UserWarning)

print("✅ Environment initialized. All dependencies are ready.")


Note: you may need to restart the kernel to use updated packages.
✅ Environment initialized. All dependencies are ready.


## 1 — Character Details

In [4]:
import json
import re
from pathlib import Path
from openai import OpenAI

# --- 1. CORE CONFIGURATION ---
CONFIG = {
    "base_url": "http://127.0.0.1:1234/v1", 
    "api_key": "lm-studio",
    "model": "loaded-model"
}

client = OpenAI(base_url=CONFIG["base_url"], api_key=CONFIG["api_key"])

class Col:
    BLUE = "\033[94m"; CYAN = "\033[96m"; GREEN = "\033[92m"
    YELLOW = "\033[93m"; RED = "\033[91m"; BOLD = "\033[1m"; END = "\033[0m"

# --- 2. REPAIRED FUNCTIONS ---
def ask_llm(system_prompt, user_content):
    """Universal LLM caller that handles the 'Type' error by using text-mode."""
    try:
        response = client.chat.completions.create(
            model=CONFIG["model"],
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_content}
            ],
            # FIXED: Changed from 'json_object' to 'text' for maximum compatibility
            response_format={"type": "text"}, 
            temperature=0.2
        )
        
        content = response.choices[0].message.content
        
        # We manually find the JSON block in case the model adds chatter
        json_match = re.search(r'\{.*\}', content, re.DOTALL)
        if json_match:
            return json.loads(json_match.group(0))
        return None
        
    except Exception as e:
        print(f"{Col.RED}LLM Error: {e}{Col.END}")
        return None

def extract_character_profile(text):
    """Uses a strong prompt to ensure we still get JSON even in text-mode."""
    system_prompt = (
        "You are a Data Extractor. Extract Name, Type, Gender, and Personality "
        "from the text. Return ONLY a valid JSON object. Do not include any "
        "introductory text or explanation."
    )
    profile = ask_llm(system_prompt, text)
    return profile if isinstance(profile, dict) else {"name": "Unknown", "type": "Unknown", "personality": "Unknown"}

# --- 3. EXECUTION LOGIC ---
DATASET_DIR = Path("dataset")
identity_keywords = ["details", "bio", "profile", "identity", "char_info"]
details_candidates = []

if DATASET_DIR.exists():
    for f in DATASET_DIR.rglob("*"):
        if any(kw in f.name.lower() for kw in identity_keywords) and f.is_file():
            details_candidates.append(f)

if not details_candidates:
    print(f"{Col.YELLOW}⚠️ No identity files found in {DATASET_DIR}.{Col.END}")
    char_profile = {"name": "Unknown"}
else:
    target_file = details_candidates[0]
    print(f"{Col.GREEN}🔍 Analyzing Identity File: {target_file.name}{Col.END}")
    
    try:
        raw_content = target_file.read_text(encoding="utf-8", errors="replace")
        char_profile = extract_character_profile(raw_content)
        char_name = char_profile.get("name", "Unknown")
        
        print(f"{Col.BOLD}✨ Character Engine Initialized:{Col.END}")
        print(f"   Name: {Col.CYAN}{char_name}{Col.END} | Type: {char_profile.get('type', 'Unknown')}")
        print(f"   Personality: {char_profile.get('personality', 'Unknown')}\n")
    except Exception as e:
        print(f"{Col.RED}❌ Logic Error: {e}{Col.END}")



🔍 Analyzing Identity File: Nayari_Details.md
✨ Character Engine Initialized:
   Name: Unknown | Type: Unknown
   Personality: Unknown



## 2 — Robust Multi-Format Parser

Handles every speaker format seen in the repo:

```
Name: text           ← plain
**Name**: text       ← bold markdown
> Name: text         ← blockquote
[Name]: text         ← bracket
```

Scene splitting triggers on: `--- END ---`, `=== break ===`, `<end>`, or **3+ blank lines**.
OOC annotations `[OOC: …]` and `(OOC …)` are stripped from message content.

In [5]:
import json
import re
import time
import hashlib
from pathlib import Path
from collections import defaultdict

# --- 1. CONFIGURATION & REGEX (Your Logic) ---
BASE_USER_ALIASES      = {"me", "you", "tiaya"}
BASE_ASSISTANT_ALIASES = {"nayari", "nayri", "aura"}

SPEAKER_RE = re.compile(
    r"""^(?:\*{1,2}|\[|>\s*)?([A-Za-z][A-Za-z0-9 _\'\-]*) (?:\*{1,2}|\])? :\s*(.*)$""",
    re.VERBOSE | re.MULTILINE
)
END_RE = re.compile(r"^[-=*]{3,}\s*(<?(end|break|scene)>?)?\s*[-=*]{0,}$", re.I)
OOC_RE = re.compile(r"\[OOC:?[^\]]*\]|\(OOC[^)]*\)", re.IGNORECASE)

class Col:
    BLUE = "\033[94m"; CYAN = "\033[96m"; GREEN = "\033[92m"
    YELLOW = "\033[93m"; RED = "\033[91m"; BOLD = "\033[1m"; END = "\033[0m"

# --- 2. THE CORE FUNCTIONS (Your Parser Logic) ---

def _strip_ooc(text: str) -> str:
    return OOC_RE.sub("", text).strip()

def _inject_scene_ends(text: str) -> str:
    return re.sub(r"(\r?\n){3,}", "\n--- END ---\n", text)

def _detect_aliases(text: str):
    counts = defaultdict(int)
    for line in text.splitlines():
        m = SPEAKER_RE.match(line.strip())
        if m: counts[m.group(1).strip().lower()] += 1
    u_extra, a_extra = set(), set()
    asst_keywords = {"nayari", "nayri", "aura", "goddess"}
    for name, cnt in counts.items():
        if cnt < 2: continue
        if any(kw in name for kw in asst_keywords): a_extra.add(name)
        else: u_extra.add(name)
    return u_extra, a_extra

def parse_chat_text(text: str, filename: str = ""):
    u_extra, a_extra = _detect_aliases(text)
    user_aliases = BASE_USER_ALIASES | u_extra
    asst_aliases = BASE_ASSISTANT_ALIASES | a_extra
    
    text = _inject_scene_ends(text)
    lines, conversations, current_messages = text.splitlines(), [], []
    current_role, buf = None, []

    def flush():
        nonlocal buf
        if current_role and buf:
            content = _strip_ooc(" ".join(" ".join(buf).split()).strip())
            if len(content) >= 10:
                current_messages.append({"role": current_role, "content": content})
        buf = []

    def save():
        if len(current_messages) >= 2:
            conversations.append({"messages": list(current_messages), "source": filename})
        current_messages.clear()

    for line in lines:
        s = line.strip()
        if not s: continue
        if (s.startswith(("#", "-", "=")) and not SPEAKER_RE.match(s)) or END_RE.match(s):
            flush(); save(); current_role = None; continue
        m = SPEAKER_RE.match(s)
        if m:
            sp, rest = m.group(1).strip().lower(), m.group(2).strip()
            if sp in user_aliases: flush(); current_role = "user"; buf = [rest]
            elif sp in asst_aliases: flush(); current_role = "assistant"; buf = [rest]
            else: 
                if current_role: buf.append(s)
        elif current_role: buf.append(s)
    flush(); save()
    return conversations

# --- 3. THE EXECUTION LOOP (The Parser) ---

processed_data = {"conversations": [], "lore": [], "failed": []}
start_time = time.time()

# Ensure all_files exists (Discovery)
DATASET_DIR = Path("dataset") # Adjust this path as needed
all_files = sorted([f for f in DATASET_DIR.rglob("*") if f.suffix.lower() in {'.json', '.md', '.txt', '.pdf'} and f.is_file()])

print(f"\n{Col.BOLD}{Col.BLUE}🧠 Intelligence Engine: Analyzing {len(all_files)} files...{Col.END}\n")

for idx, path in enumerate(all_files, 1):
    rel = str(path.relative_to(DATASET_DIR))
    # Truncate long paths for the UI
    display_name = (rel[:37] + '..') if len(rel) > 40 else rel
    ext = path.suffix.lower()
    prefix = f"{Col.BOLD}[{idx:03}/{len(all_files)}]{Col.END}"
    
    try:
        # JSON PROCESSING
        if ext == ".json":
            data = json.loads(path.read_text(encoding="utf-8", errors="replace"))
            # Logic: is it lore?
            if "lore" in rel.lower() or "world" in rel.lower() or "details" in rel.lower():
                processed_data["lore"].append({"source": rel, "content": data})
                print(f"{prefix} 📜 {Col.YELLOW}LORE-JSON{Col.END}  | {display_name:<40} | {Col.YELLOW}Data Tree{Col.END}")
            else:
                convs = []
                if isinstance(data, list): convs = [{"messages": c["messages"], "source": rel} for c in data if "messages" in c]
                elif isinstance(data, dict):
                    if "conversations" in data: convs = data["conversations"]
                    elif "messages" in data: convs = [data]
                processed_data["conversations"].extend(convs)
                print(f"{prefix} 💬 {Col.CYAN}CHAT-JSON{Col.END}  | {display_name:<40} | {Col.CYAN}{len(convs)} convs{Col.END}")

        # TEXT / MD PROCESSING
        elif ext in {".txt", ".md"}:
            raw_text = path.read_text(encoding="utf-8", errors="replace")
            # Logic: is it lore or chat?
            if "lore" in rel.lower() or "details" in rel.lower():
                processed_data["lore"].append({"source": rel, "text": raw_text})
                print(f"{prefix} 📝 {Col.YELLOW}TXT-LORE{Col.END}   | {display_name:<40} | {Col.YELLOW}{len(raw_text.split()):,} words{Col.END}")
            else:
                convs = parse_chat_text(raw_text, rel)
                processed_data["conversations"].extend(convs)
                print(f"{prefix} 📝 {Col.CYAN}TXT-CHAT{Col.END}   | {display_name:<40} | {Col.CYAN}{len(convs)} scenes{Col.END}")

        # PDF PROCESSING
        elif ext == ".pdf":
            import pdfplumber
            with pdfplumber.open(path) as pdf:
                text = "\n".join([p.extract_text() or "" for p in pdf.pages])
            convs = parse_chat_text(text, rel)
            if convs:
                processed_data["conversations"].extend(convs)
                print(f"{prefix} 📘 {Col.CYAN}PDF-CHAT{Col.END}   | {display_name:<40} | {Col.CYAN}{len(convs)} scenes{Col.END}")
            else:
                processed_data["lore"].append({"source": rel, "text": text})
                print(f"{prefix} 📘 {Col.YELLOW}PDF-LORE{Col.END}   | {display_name:<40} | {Col.YELLOW}Knowledge{Col.END}")

    except Exception as exc:
        print(f"{prefix} ❌ {Col.RED}FAILED{Col.END}     | {display_name:<40} | {Col.RED}{type(exc).__name__}{Col.END}")
        processed_data["failed"].append(rel)

# --- 4. FINAL SMART BREAKDOWN ---

total_convs = len(processed_data['conversations'])
total_msgs = sum(len(c.get("messages", [])) for c in processed_data["conversations"])
total_lore = len(processed_data['lore'])
elapsed = time.time() - start_time

print(f"\n{Col.BOLD}{'═'*75}{Col.END}")
print(f"{Col.BOLD}📊 INTELLIGENCE ENGINE: FINAL DATASET BREAKDOWN{Col.END}")
print(f"{Col.BOLD}{'═'*75}{Col.END}")
print(f"⏱️  Processing Time:   {Col.BOLD}{elapsed:.2f} seconds{Col.END}")
print(f"💬 Total Conversations: {Col.GREEN}{total_convs:,}{Col.END}")
print(f"💬 Total Messages:      {Col.GREEN}{total_msgs:,}{Col.END}")
print(f"📜 Lore/Knowledge:      {Col.YELLOW}{total_lore:,} entries{Col.END}")
print(f"❌ Failed Files:        {Col.RED}{len(processed_data['failed'])}{Col.END}")

if total_convs > 0:
    avg = total_msgs / total_convs
    print(f"📈 Content Density:     {Col.CYAN}{avg:.1f} messages per conversation{Col.END}")

print(f"{Col.BOLD}{'═'*75}{Col.END}")


🧠 Intelligence Engine: Analyzing 294 files...

[001/294] 📝 TXT-CHAT   | set-1/md files/chats/Nayari_chat_1.md    | 1 scenes
[002/294] 📝 TXT-CHAT   | set-1/md files/chats/Nayari_chat_10.md   | 4 scenes
[003/294] 📝 TXT-CHAT   | set-1/md files/chats/Nayari_chat_11.md   | 4 scenes
[004/294] 📝 TXT-CHAT   | set-1/md files/chats/Nayari_chat_12.md   | 4 scenes
[005/294] 📝 TXT-CHAT   | set-1/md files/chats/Nayari_chat_13.md   | 4 scenes
[006/294] 📝 TXT-CHAT   | set-1/md files/chats/Nayari_chat_14.md   | 4 scenes
[007/294] 📝 TXT-CHAT   | set-1/md files/chats/Nayari_chat_15.md   | 1 scenes
[008/294] 📝 TXT-CHAT   | set-1/md files/chats/Nayari_chat_16.md   | 1 scenes
[009/294] 📝 TXT-CHAT   | set-1/md files/chats/Nayari_chat_17.md   | 1 scenes
[010/294] 📝 TXT-CHAT   | set-1/md files/chats/Nayari_chat_18.md   | 1 scenes
[011/294] 📝 TXT-CHAT   | set-1/md files/chats/Nayari_chat_19.md   | 1 scenes
[012/294] 📝 TXT-CHAT   | set-1/md files/chats/Nayari_chat_2.md    | 1 scenes
[013/294] 📝 TXT-CHAT   | set

## 3 — Parse All Source Files (.md / .txt / .pdf / .json)

In [6]:
import json
import time
import re
from pathlib import Path

# --- 1. SETTINGS ---
# Set this to the EXACT name of your folder
FOLDER_NAME = "dataset" 
DATASET_DIR = Path(FOLDER_NAME)

# Extensions to look for (Case-Insensitive)
EXTENSIONS = {".md", ".txt", ".pdf", ".json"}

class Col:
    BLUE = "\033[94m"; CYAN = "\033[96m"; GREEN = "\033[92m"
    YELLOW = "\033[93m"; RED = "\033[91m"; BOLD = "\033[1m"; END = "\033[0m"

# --- 2. FOLDER AUDIT ---
print(f"{Col.BOLD}🔍 Folder Audit:{Col.END}")
if not DATASET_DIR.exists():
    print(f"  {Col.RED}❌ ERROR: Folder '{FOLDER_NAME}' not found!{Col.END}")
    print(f"  Current location: {Path.cwd()}")
    # List everything in current directory to help you find it
    print(f"  I see these folders here: {[d.name for d in Path('.').iterdir() if d.is_dir()]}")
else:
    # Check if the folder is empty
    all_items = list(DATASET_DIR.rglob("*"))
    print(f"  {Col.GREEN}✅ Folder found.{Col.END} Total items inside: {len(all_items)}")

# --- 3. RE-DEFINING MISSING HELPERS (To prevent NameErrors) ---
def is_lore_file(file_path):
    path_str = str(file_path).lower()
    return any(kw in path_str for kw in ['lore', 'world', 'bio', 'details', 'info'])

def parse_chat_text(text, source_name):
    # Matches Name: Message, **Name**: Message, etc.
    pattern = re.compile(r"^(?:\*?\*?|>?\s*\[?)([A-Za-z][A-Za-z0-9 _]*)(?:\]?|:|\*?\*?)\s*:\s*(.*)", re.MULTILINE)
    messages = []
    for m in pattern.finditer(text):
        messages.append({"role": m.group(1).strip(), "content": m.group(2).strip()})
    return [{"source": source_name, "messages": messages}] if messages else []

# --- 4. THE MAIN ENGINE ---
all_files = [f for f in DATASET_DIR.rglob("*") if f.suffix.lower() in EXTENSIONS and f.is_file()]

processed_data = {"conversations": [], "lore": [], "failed": []}
start_time = time.time()

print(f"\n{Col.BOLD}{Col.BLUE}🧠 Intelligence Engine: Analyzing {len(all_files)} files...{Col.END}\n")

for idx, path in enumerate(all_files, 1):
    rel = str(path.relative_to(DATASET_DIR))
    ext = path.suffix.lower()
    prefix = f"{Col.BOLD}[{idx}/{len(all_files)}]{Col.END}"
    
    try:
        # JSON
        if ext == ".json":
            data = json.loads(path.read_text(encoding="utf-8", errors="replace"))
            if is_lore_file(path):
                processed_data["lore"].append({"source": rel, "content": data})
                print(f"{prefix} 📜 {Col.YELLOW}LORE-JSON{Col.END}  | {rel[:40]}")
            else:
                processed_data["conversations"].append({"messages": data, "source": rel})
                print(f"{prefix} 💬 {Col.CYAN}CHAT-JSON{Col.END}  | {rel[:40]}")

        # TEXT/MD
        elif ext in {".md", ".txt"}:
            raw_text = path.read_text(encoding="utf-8", errors="replace")
            if is_lore_file(path):
                processed_data["lore"].append({"source": rel, "text": raw_text})
                print(f"{prefix} 📝 {Col.YELLOW}TXT-LORE{Col.END}   | {rel[:40]}")
            else:
                convs = parse_chat_text(raw_text, rel)
                processed_data["conversations"].extend(convs)
                print(f"{prefix} 📝 {Col.CYAN}TXT-CHAT{Col.END}   | {rel[:40]}")

    except Exception as e:
        print(f"{prefix} ❌ {Col.RED}FAILED{Col.END}     | {rel[:40]} | {str(e)[:30]}")
        processed_data["failed"].append(rel)

# --- 5. SUMMARY ---
elapsed = time.time() - start_time
print(f"\n{Col.BOLD}📊 FINAL DATASET REPORT{Col.END}")
print(f"══════════════════════════════════════════")
print(f"⏱️  Time:        {elapsed:.2f}s")
print(f"💬 Dialogues:   {Col.GREEN}{len(processed_data['conversations'])}{Col.END}")
print(f"📜 Lore:        {Col.YELLOW}{len(processed_data['lore'])}{Col.END}")
print(f"❌ Failed:      {Col.RED}{len(processed_data['failed'])}{Col.END}")
print(f"══════════════════════════════════════════")

🔍 Folder Audit:
  ✅ Folder found. Total items inside: 305

🧠 Intelligence Engine: Analyzing 294 files...

[167/294] 📝 TXT-CHAT   | set-1/md files/chats/Nayari_chat_1.md
[168/294] 📝 TXT-CHAT   | set-1/md files/chats/Nayari_chat_10.md
[169/294] 📝 TXT-CHAT   | set-1/md files/chats/Nayari_chat_11.md
[170/294] 📝 TXT-CHAT   | set-1/md files/chats/Nayari_chat_12.md
[171/294] 📝 TXT-CHAT   | set-1/md files/chats/Nayari_chat_13.md
[172/294] 📝 TXT-CHAT   | set-1/md files/chats/Nayari_chat_14.md
[173/294] 📝 TXT-CHAT   | set-1/md files/chats/Nayari_chat_15.md
[174/294] 📝 TXT-CHAT   | set-1/md files/chats/Nayari_chat_16.md
[175/294] 📝 TXT-CHAT   | set-1/md files/chats/Nayari_chat_17.md
[176/294] 📝 TXT-CHAT   | set-1/md files/chats/Nayari_chat_18.md
[177/294] 📝 TXT-CHAT   | set-1/md files/chats/Nayari_chat_19.md
[178/294] 📝 TXT-CHAT   | set-1/md files/chats/Nayari_chat_2.md
[179/294] 📝 TXT-CHAT   | set-1/md files/chats/Nayari_chat_20.md
[180/294] 📝 TXT-CHAT   | set-1/md files/chats/Nayari_chat_21.md


## 4 — Lore → Training Conversations

Each PDF lore section is chunked into paragraphs.
Each chunk is paired with a varied natural-language prompt so the model learns
to recall Nayari's lore organically. Multiple prompts per chunk for augmentation.

In [8]:
import json
import time
import re
from pathlib import Path
from openai import OpenAI  # pip install openai

# --- PROVIDER CONFIGURATION ---
PROVIDER_CONFIG = {
    "base_url": "http://127.0.0.1:1234/v1", # Update to the base url 
    "api_key": "lm-studio",
    "model": "loaded-model" 
}

client = OpenAI(
    base_url=PROVIDER_CONFIG["base_url"],
    api_key=PROVIDER_CONFIG["api_key"]
)

class Col:
    BLUE, CYAN, GREEN, YELLOW, RED, BOLD, END = "\033[94m", "\033[96m", "\033[92m", "\033[93m", "\033[91m", "\033[1m", "\033[0m"

# --- LLM CORE LOGIC ---
def call_llm(system_msg, user_msg):
    """Universal wrapper for any OpenAI-compatible API."""
    try:
        response = client.chat.completions.create(
            model=PROVIDER_CONFIG["model"],
            messages=[
                {"role": "system", "content": system_msg},
                {"role": "user", "content": user_msg}
            ],
            response_format={"type": "json_object"}, # Supported by most modern models
            temperature=0.1
        )
        return json.loads(response.choices[0].message.content)
    except Exception as e:
        print(f"  {Col.RED}LLM Error: {e}{Col.END}")
        return None

# --- PROCESSING TASKS ---
def process_lore_with_llm(chunk, source_name):
    """Turns dry lore into a Q&A pair."""
    system_prompt = """
    You are a Dataset Architect. Convert the provided LORE text into a single Conversation Pair.
    1. Clean any PDF/OCR artifacts.
    2. Create a 'user' question that naturally asks for this information.
    3. The 'assistant' response should be the cleaned text.
    Return ONLY JSON: {"user": "...", "assistant": "..."}
    """
    result = call_llm(system_prompt, chunk)
    if result:
        return {
            "messages": [
                {"role": "user", "content": result["user"]},
                {"role": "assistant", "content": result["assistant"]}
            ],
            "metadata": {"source": source_name, "type": "synthetic_lore"}
        }
    return None

def process_chat_with_llm(raw_chat, source_name):
    """Normalizes messy chat transcripts into structured JSON."""
    system_prompt = """
    Convert this messy chat transcript into a structured JSON list of messages.
    Normalize names (e.g., 'Nayari_123' -> 'Nayari'). 
    Return ONLY JSON: {"messages": [{"role": "name", "content": "..."}, ...]}
    """
    result = call_llm(system_prompt, raw_chat)
    if result:
        res = {"messages": result["messages"], "metadata": {"source": source_name, "type": "cleaned_chat"}}
        return res
    return None

def intelligence_engine(all_files):
    processed_dataset = []
    print(f"\n{Col.BOLD}{Col.BLUE}🧠 Intelligence Engine Active ({PROVIDER_CONFIG['model']}){Col.END}\n")

    for idx, path in enumerate(all_files, 1):
        rel = str(path.name)
        ext = path.suffix.lower()
        print(f"[{idx}/{len(all_files)}] Processing {Col.CYAN}{rel}{Col.END}...")
        try:
            if ext in [".md", ".txt"]:
                content = path.read_text(encoding="utf-8", errors="replace")
                if "lore" in rel.lower() or len(re.findall(r'\b(is|was|the|history)\b', content[:200])) > 5:
                    chunks = [content[i:i+1500] for i in range(0, len(content), 1500)]
                    for chunk in chunks:
                        data = process_lore_with_llm(chunk, rel)
                        if data: processed_dataset.append(data)
                else:
                    data = process_chat_with_llm(content, rel)
                    if data: processed_dataset.append(data)
            elif ext == ".json":
                raw_json = path.read_text(encoding="utf-8")
                processed_dataset.append({"raw_json": json.loads(raw_json), "metadata": {"source": rel}})
        except Exception as e:
            print(f"  {Col.RED}Failed file {rel}: {e}{Col.END}")
    return processed_dataset

## 5 — Deduplication & Quality Filter

In [9]:
import hashlib
import json

def smart_deduplicate(conversations):
    unique_convs = []
    seen_hashes = set()
    for conv in conversations:
        content_str = "".join([m["content"] for m in conv["messages"]])
        fingerprint = hashlib.sha256(content_str.encode('utf-8')).hexdigest()
        if fingerprint not in seen_hashes:
            unique_convs.append(conv)
            seen_hashes.add(fingerprint)
    return unique_convs

def repair_oversized_message(role, content, char_name):
    repair_prompt = f"""
    This message is too long for training. 
    If it's Lore: Summarize it into a concise paragraph.
    If it's Dialogue: Split it into 3 shorter messages.
    Return ONLY JSON: {{"repaired_content": "..."}}
    """
    # NOTE: Notebook uses 'call_small_llm' which must be defined or swapped for 'call_llm'
    result = call_llm(repair_prompt, content[:2000]) 
    return result.get("repaired_content", content) if result else content

def finalize_dataset(processed_data, char_name):
    print(f"\n{Col.BOLD}🛠️  Refining & Optimizing Dataset...{Col.END}")
    all_raw = processed_data['conversations']
    lore_raw = processed_data['lore']
    initial_count = len(all_raw) + len(lore_raw)
    optimized_convs = smart_deduplicate(all_raw + lore_raw)
    final_count = len(optimized_convs)
    
    print(f"  {Col.GREEN}✓{Col.END} Deduplication: {initial_count} -> {final_count} (Removed {initial_count - final_count})")

    WARN_CHARS = 2500 
    repaired_count = 0
    for conv in optimized_convs:
        for m in conv["messages"]:
            if len(m["content"]) > WARN_CHARS:
                print(f"  {Col.YELLOW}⚡ Auto-Repairing long message ({len(m['content'])} chars)...{Col.END}")
                m["content"] = repair_oversized_message(m["role"], m["content"], char_name)
                repaired_count += 1

    stats = {
        "md": len([c for c in optimized_convs if str(c.get('source','')).endswith('.md')]),
        "pdf": len([c for c in optimized_convs if str(c.get('source','')).endswith('.pdf')]),
        "json": len([c for c in optimized_convs if str(c.get('source','')).endswith('.json')]),
        "txt": len([c for c in optimized_convs if str(c.get('source','')).endswith('.txt')])
    }
    return optimized_convs

## 6 — Preview & Statistics Dashboard

In [10]:
import hashlib
from collections import defaultdict

def get_content_hash(messages):
    full_str = "".join([f"{m['role']}:{m['content']}" for m in messages])
    return hashlib.sha256(full_str.encode('utf-8')).hexdigest()

def smart_optimize_dataset(processed_data, char_name):
    print(f"\n{Col.BOLD}{Col.BLUE}🛠️  Running Smart Optimizer...{Col.END}")
    all_raw = processed_data.get('conversations', []) + processed_data.get('lore', [])
    by_ext = defaultdict(list)
    for conv in all_raw:
        ext = Path(conv.get('source', 'unknown.txt')).suffix.lower() or '.txt'
        by_ext[ext].append(conv)

    unique_convs = []
    seen_hashes = set()
    duplicates_removed = 0
    for conv in all_raw:
        h = get_content_hash(conv.get('messages', []))
        if h not in seen_hashes:
            unique_convs.append(conv)
            seen_hashes.add(h)
        else:
            duplicates_removed += 1

    MAX_DIALOGUE_LENGTH = 2800
    final_cleaned = []
    repair_count = 0
    for conv in unique_convs:
        new_messages = []
        for msg in conv.get("messages", []):
            if len(msg["content"]) > MAX_DIALOGUE_LENGTH:
                repair_prompt = f"Summarize this into 2-3 concise paragraphs for {char_name}."
                repaired = call_llm(repair_prompt, msg["content"][:4000])
                if repaired and "repaired_content" in repaired:
                    msg["content"] = repaired["repaired_content"]
                    repair_count += 1
                else:
                    msg["content"] = msg["content"][:MAX_DIALOGUE_LENGTH] + "... [Truncated]"
            new_messages.append(msg)
        conv["messages"] = new_messages
        final_cleaned.append(conv)
    return final_cleaned

## 7 — Export JSON

In [12]:
import json
import hashlib
from datetime import datetime, timezone
from pathlib import Path

def get_dataset_hash(conversations, lore):
    content_blob = json.dumps(conversations, sort_keys=True) + json.dumps(lore, sort_keys=True)
    return hashlib.sha256(content_blob.encode()).hexdigest()

def bump_version(old_version, change_type="patch"):
    try:
        major, minor, patch = map(int, old_version.split('.'))
        if change_type == "major": major += 1; minor = 0; patch = 0
        elif change_type == "minor": minor += 1; patch = 0
        else: patch += 1
        return f"{major}.{minor}.{patch}"
    except: return "1.0.0"

def save_smart_dataset(all_conversations, char_profile, output_path, model_name):
    print(f"\n{Col.BOLD}{Col.BLUE}💾 Running Smart Versioning & Export...{Col.END}")
    prev_version, prev_hash, prev_engine = "1.0.0", "", ""
    old_data = {} # Fixed: ensure old_data is defined
    
    if output_path.exists():
        try:
            with open(output_path, "r", encoding="utf-8") as f:
                old_data = json.load(f)
                prev_version = old_data.get("dataset_metadata", {}).get("version", "1.0.0")
                prev_hash = old_data.get("dataset_metadata", {}).get("content_hash", "")
                prev_engine = old_data.get("dataset_metadata", {}).get("engine", "")
        except: pass

    current_hash = get_dataset_hash(all_conversations, char_profile.get("lore_data", []))
    total_msgs = sum(len(c.get("messages", [])) for c in all_conversations)
    est_tokens = sum(len(m["content"]) for c in all_conversations for m in c["messages"]) // 4
    
    if current_hash == prev_hash:
        new_version = prev_version
    else:
        if model_name != prev_engine and prev_engine != "":
            new_version = bump_version(prev_version, "major")
        elif len(all_conversations) != old_data.get("dataset_metadata", {}).get("stats", {}).get("total_conversations", 0):
            new_version = bump_version(prev_version, "minor")
        else:
            new_version = bump_version(prev_version, "patch")

    dataset_json = {
        "dataset_metadata": {
            "version": new_version,
            "content_hash": current_hash,
            "generated_at": datetime.now(timezone.utc).isoformat(),
            "engine": model_name,
            "stats": {
                "total_conversations": len(all_conversations),
                "total_messages": total_msgs,
                "estimated_tokens": est_tokens
            }
        },
        "character_profile": char_profile,
        "conversations": all_conversations
    }
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(dataset_json, f, ensure_ascii=False, indent=2)


## 8 — Upload to Kaggle 

### 🚀 Option A: Automatic Upload (Kaggle API)
*Best for: Developers and frequent updates.*

1. **Setup API Key:** Ensure your `kaggle.json` file is located in:
   * **Linux/Mac:** `~/.kaggle/`
   * **Windows:** `C:\Users\<User>\.kaggle\`
   * *Tip: Run `chmod 600 ~/.kaggle/kaggle.json` on Linux/Mac for security.*

2. **Initialize Metadata:** Generate the required `dataset-metadata.json` file:
   ```bash
   kaggle datasets init -p /path/to/dataset
   ```
   *Open the file and edit the `title` and `id` (slug).*

3. **Upload Command:**
   ```bash
   # For a new dataset:
   kaggle datasets create -p .
   
   # To update an existing dataset:
   kaggle datasets version -p . -m "Updated lore and chat logs"
   ```

---

### 🖱️ Option B: Manual Upload (Web Interface)
*Best for: One-time uploads or those who prefer a GUI.*

1. **Navigate to Kaggle:** Go to [kaggle.com/datasets](https://www.kaggle.com/datasets) and log in.
2. **Start Creation:** Click the **+ New Dataset** button (top right).
3. **Upload Files:** 
   * **Drag & Drop:** Your CSVs, JSONs, or ZIP files into the upload box.
   * **Title:** Give your dataset a clear name (e.g., "Nayari Lore & Chat Logs").
4. **Settings:**
   * **Privacy:** Choose **Private** (default) or **Public**.
   * **License:** Select a license (e.g., CC BY-SA 4.0).
5. **Finish:** Click **Create** at the bottom of the window.

---

### 💡 Pro-Tips for Better Datasets
* **Zip Large Files:** If your dataset is >100MB, zip it before uploading to speed up the process.
* **Documentation:** Use the **"Description"** tab on Kaggle to explain the lore context; this helps other users understand your data.
* **Metadata ID:** When using the API, the `id` in your `.json` must be in the format `username/dataset-slug`.